# اجرای پروژه mT5 فارسی در Colab
قبل از شروع: منوی **Runtime → Change runtime type → T4 GPU**

کد و داده مستقیم از گیت‌هاب خوانده می‌شوند (بدون Google Drive).

⚠️ مدل‌ها روی دیسک موقت Colab ذخیره می‌شوند و با قطع شدن Colab پاک می‌شوند؛ همه مراحل را در یک نشست اجرا کنید و در پایان نتایج را دانلود کنید.

## ۱. دریافت کد و داده از گیت‌هاب

In [ ]:
import os, glob, zipfile, shutil
if not os.path.isdir('/content/persian-mt5-summarization-sentiment'):
    !git clone https://github.com/zph272/persian-mt5-summarization-sentiment.git
%cd /content/persian-mt5-summarization-sentiment
!git pull

# unzip the processed data (each file was zipped separately)
os.makedirs('data/processed', exist_ok=True)
for zp in glob.glob('*.zip') + glob.glob('data/**/*.zip', recursive=True):
    if os.path.basename(zp) != 'data.csv.zip':
        zipfile.ZipFile(zp).extractall('data/processed')
for f in glob.glob('data/processed/**/*.csv', recursive=True):
    if os.path.dirname(f) != 'data/processed':
        shutil.move(f, 'data/processed/' + os.path.basename(f))
if not os.path.exists('data/processed/joint_clean.csv') and os.path.exists('data/processed/joint_train.csv'):
    shutil.copy('data/processed/joint_train.csv', 'data/processed/joint_clean.csv')
need = ['summ_clean.csv', 'sent_clean.csv', 'joint_train.csv', 'joint_val.csv', 'joint_test.csv']
missing = [f for f in need if not os.path.exists('data/processed/' + f)]
print('❌ missing:', missing) if missing else print('✅ all data files are ready')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## ۲. نصب کتابخانه‌ها

In [ ]:
!pip install -q rouge_score sentencepiece arabic-reshaper python-bidi

## ۳. مرحله ۱ — آموزش خلاصه‌سازی

In [ ]:
!python 4_train_stage1_summ.py

## ۴. مرحله ۲ — آموزش تحلیل احساس (از مدل مرحله ۱)
حدود ۲۰ تا ۳۰ دقیقه.

In [ ]:
!python 5_train_stage2_sent.py

## ۵. مرحله ۳ — آموزش مشترک (از مدل مرحله ۲)
چند ساعت.

In [ ]:
!python 6_train_stage3_joint.py

## ۶. ارزیابی روی داده تست
خط پایه‌ها، سپس هر مدل. ارزیابی خلاصه‌سازی روی ۵٬۵۹۳ خبر زمان‌بر است (حدود ۳۰ تا ۶۰ دقیقه برای هر مدل).

In [ ]:
!python 7_evaluate.py --baselines
!python 7_evaluate.py --model summ_final  --tasks summ,loss
!python 7_evaluate.py --model sent_final  --tasks summ,sent
!python 7_evaluate.py --model joint_final --tasks summ,sent,loss

## ۷. Ablation (اختیاری ولی برای مقاله مفید)
- `sent_only`: فقط احساس، از mT5-base (حدود ۳۰ دقیقه)
- `joint_only`: آموزش مشترک بدون مراحل ۱ و ۲ (چند ساعت)

In [ ]:
!python 9_ablation.py --only sent

In [ ]:
# !python 9_ablation.py --only joint

## ۸. نمودارها و جدول نتایج

In [ ]:
!python 8_plot_results.py
from IPython.display import Image, display, Markdown
display(Markdown(open('outputs/results/results_table.md', encoding='utf-8').read()))
import glob
for f in sorted(glob.glob('outputs/figures/*.png')):
    print(f); display(Image(f, width=600))

## ۹. دانلود نتایج (قبل از بستن Colab)

In [ ]:
!zip -qr results.zip outputs/results outputs/figures
from google.colab import files
files.download('results.zip')